In [ ]:
import random
import csv
import itertools
import math
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)

prompts_per_diff_level = 128

# Create the prompts and the boxes

In [ ]:
# Step 1: upload the csv file containing the complex compositions prompts
from google.colab import files
uploaded = files.upload()

# Load the uploaded CSV into a DataFrame
filename = list(uploaded.keys())[0]  # get the uploaded file name
complex_prompts_df = pd.read_csv(filename)

COCO classes extended to a total of 128


In [ ]:
obj = ['rose', 'oak', 'beetle', 'skyscraper', 'tree', 'baby', 'bed', 'lamp', 'dog', 'laptop', 'bicycle', 'person',
       'car', 'bus', 'cat', 'book', 'chair', 'boy', 'couch', 'table', 'plant', 'toilet', 'cellphone', 'microwave',
       'sheep', 'boat', 'banana', 'stop sign', 'donut', 'cow', 'clock', 'bottle', 'umbrella', 'bird', 'guitar',
       'toothbrush', 'parking meter', 'bench', 'platypus', 'keyboard', 'baseball bat', 'vase', 'surfboard', 'tiger',
       'train', 'flower', 'sandwich', 'spoon', 'pizza', 'carrot', 'teddy bear', 'hot-dog', 'skateboard', 'kite', 'broom',
       'apple', 'handbag', 'horse', 'snowboard', 'giraffe', 'tie', 'shower', 'traffic light', 'bear', 'toaster', 'knife',
       'baseball glove', 'crocodile', 'suitcase', 'fork', 'cake', 'cup', 'bowl', 'hair drier', 'elephant', 'mouse',
       'mushroom', 'motorcycle', 'turtle', 'tennis racket', 'truck', 'zebra', 'fire hydrant', 'oven', 'sink', 'frisbee',
       'hat', 'ruler', 'shoe', 'ball', 'candle', 'ladder', 'charger', 'mug', 'tape', 'shirt', 'pillow', 'pan', 'plate',
       'shampoo', 'hammer', 'blender', 'basket', 'screwdriver', 'wallet', 'bin', 'leaf', 'bucket', 'monitor', 'watch',
       'flashlight', 'sock', 'door', 'scarf', 'speaker', 'desk', 'backpack', 'printer', 'remote', 'glass', 'curtain',
       'toolbox', 'drill', 'notebook', 'television', 'soap', 'ring', 'refrigerator']

obj_with_articles = [
    'a rose', 'an oak', 'a beetle', 'a skyscraper', 'a tree', 'a baby', 'a bed', 'a lamp', 'a dog', 'a laptop',
    'a bicycle', 'a person', 'a car', 'a bus', 'a cat', 'a book', 'a chair', 'a boy', 'a couch', 'a table',
    'a plant', 'a toilet', 'a cellphone', 'a microwave', 'a sheep', 'a boat', 'a banana', 'a stop sign',
    'a donut', 'a cow', 'a clock', 'a bottle', 'an umbrella', 'a bird', 'a guitar', 'a toothbrush', 'a parking meter',
    'a bench', 'a platypus', 'a keyboard', 'a baseball bat', 'a vase', 'a surfboard', 'a tiger', 'a train', 'a flower',
    'a sandwich', 'a spoon', 'a pizza', 'a carrot', 'a teddy bear', 'an hot-dog', 'a skateboard', 'a kite', 'a broom',
    'an apple', 'a handbag', 'a horse', 'a snowboard', 'a giraffe', 'a tie', 'a shower', 'a traffic light', 'a bear',
    'a toaster', 'a knife', 'a baseball glove', 'a crocodile', 'a suitcase', 'a fork', 'a cake', 'a cup', 'a bowl',
    'a hair drier', 'an elephant', 'a mouse', 'a mushroom', 'a motorcycle', 'a turtle', 'a tennis racket', 'a truck',
    'a zebra', 'a fire hydrant', 'an oven', 'a sink', 'a frisbee', 'a hat', 'a ruler', 'a shoe', 'a ball', 'a candle',
    'a ladder', 'a charger', 'a mug', 'a tape', 'a shirt', 'a pillow', 'a pan', 'a plate', 'a shampoo', 'a hammer',
    'a blender', 'a basket', 'a screwdriver', 'a wallet', 'a bin', 'a leaf', 'a bucket', 'a monitor', 'a watch',
    'a flashlight', 'a sock', 'a door', 'a scarf', 'a speaker', 'a desk', 'a backpack', 'a printer', 'a remote',
    'a glass', 'a curtain', 'a toolbox', 'a drill', 'a notebook', 'a television', 'a soap', 'a ring', 'a refrigerator'
]


In [ ]:
colors = ['black', 'blue', 'brown', 'gray', 'green', 'pink', 'purple', 'red', 'white', 'yellow', 'orange']

spatial_relations = ['above', 'below', 'beside', 'far from', 'near', 'next to', 'on', 'over', 'to the left of', 'to the right of', 'under']



attributes = [
    'aggressive', 'black', 'blue', 'bright', 'clean', 'crowded', 'dark', 'fast', 'fluffy', 'fuzzy', 'green', 'happy', 'large', 'pink', 'red', 'rotten',
    'rough', 'shiny', 'short', 'silver', 'small', 'smooth', 'snowy', 'soft', 'tall', 'warm', 'white', 'wooden', 'yellow'
]

For one object prompts, the size of the box can be bigger since there is no risk of overlapping.

The more objects are in the prompt, the smaller the boxes should be to avoid overlapping.

In [ ]:
# image size
IMAGE_SIZE = 512

# box size ranges depending on number of objects
NO_OVERLAPPING_RANGES_OLD = {
    1: (150, 350),  # larger boxes
    2: (120, 250),
    3: (100, 180),
    4: (80, 150),   # smaller boxes
}

NO_OVERLAPPING_RANGES = {
    1: (150, 500),  # larger boxes
    2: (120, 250),
    3: (100, 180),
    4: (80, 150),   # smaller boxes
}

**Overlap check** ⬇️:

Each box is defined by:
*   top-left corner → (x_min, y_min)
*   bottom-right corner → (x_max, y_max)

Two boxes **do NOT overlap** if one of these is true:
* Box1 is completely to the left of Box2 → x1_max < x2_min
* Box1 is completely to the right of Box2 → x2_max < x1_min
* Box1 is completely above Box2 → y1_max < y2_min
* Box1 is completely below Box2 → y2_max < y1_min

When all of these are false, the two boxes overlap.*italicised text*

In [ ]:
# overlap check
def boxes_overlap(box1, box2):
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    return not (x1_max < x2_min or x2_max < x1_min or y1_max < y2_min or y2_max < y1_min)


**Random box generation tools** ⬇️:

First width and height of the box are generated randomly between the correct sizes based on the number of object there will be in the prompt.

Then the box is positioned inside the space by choosing randomly the top-left corner according to the height and width.

In [ ]:
# generate box
def generate_random_box(min_size, max_size):
    width = random.randint(min_size, max_size)
    height = random.randint(min_size, max_size)
    x1 = random.randint(0, IMAGE_SIZE - width)
    y1 = random.randint(0, IMAGE_SIZE - height)
    x3 = x1 + width
    y3 = y1 + height
    return (x1, y1, x3, y3)

In [ ]:
# build prompt string: fixed structure based on the number of objects
def build_prompt(objs):
    if len(objs) == 1:
        return objs[0]
    elif len(objs) == 2:
        return f'{objs[0]} and {objs[1]}'
    elif len(objs) == 3:
        return f'{objs[0]}, {objs[1]} and {objs[2]}'
    elif len(objs) == 4:
        return f'{objs[0]}, {objs[1]}, {objs[2]} and {objs[3]}'

In [ ]:
# generate N non-overlapping boxes
def generate_non_overlapping_boxes(num_boxes, min_size, max_size):
    boxes = []
    attempts = 0
    max_attempts = 1000
    while len(boxes) < num_boxes and attempts < max_attempts:
        new_box = generate_random_box(min_size, max_size)
        if all(not boxes_overlap(new_box, existing_box) for existing_box in boxes):
            boxes.append(new_box)
        attempts += 1
    # if after 1000 times still not overlapping bounging boxes were generated, raise an error
    if len(boxes) < num_boxes:
        raise RuntimeError("Failed to generate non-overlapping boxes after many attempts.")
    return boxes

### Object binding functions

With no overlapping bboxes

In [ ]:
def generate_csv_object_binding(output_path, id_counter):
    mode = 'w' if id_counter == 0 else 'a'

    with open(output_path, mode=mode, newline='') as csv_file:
        writer = csv.writer(csv_file)

        # Write header only if starting fresh
        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt', 'obj1', 'bbox1', 'obj2', 'bbox2', 'obj3', 'bbox3', 'obj4', 'bbox4'])


        # for each number of objects
        for num_objs in [1, 2, 3, 4]:
            print(f"Generating prompts with {num_objs} objects...")
            min_size, max_size = NO_OVERLAPPING_RANGES[num_objs]

            # prepare unique combinations of objects: they should not repeat in the same prompt
            if num_objs == 1:
                combinations = [[obj] for obj in obj_with_articles]
            else:
                # itertools.combinations only generates combinations of unique objects (no repetition within a tuple)
                combinations = list(itertools.combinations(obj_with_articles, num_objs))

            # Shuffle combinations to ensure that when we repeat and truncate them to 512 prompts,
            # the resulting prompts are more varied and not biased toward always starting with the same combinations.
            random.shuffle(combinations)

            # how many times do we need to repeat combinations to reach 512 prompts per number of objects
            total_needed = prompts_per_diff_level
            repeats_needed = math.ceil(total_needed / len(combinations))

            used_combinations = (combinations * repeats_needed)[:total_needed]

            # tqdm is to create the progress bar
            for objs in tqdm(used_combinations):
                boxes = generate_non_overlapping_boxes(num_objs, min_size, max_size)

                id_str = str(id_counter).zfill(4)
                category = 'object_binding'
                prompt = build_prompt(objs)
                row = [id_str, category, prompt]

                # pair together objects + boxes
                for obj, box in itertools.zip_longest(objs, boxes, fillvalue=''):
                    row.append(obj)
                    # write the corresponding bounding box, or empty string if there is no box
                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                # fill the row with empty string for prompts with less than 4 objects
                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1

    return id_counter

### Color binding functions

Still no overlapping bounding boxes

In [ ]:
def build_color_prompt(objs, colors_sampled):
    colored_objs = []
    for obj, color in zip(objs, colors_sampled):
        article, noun = obj.split(' ', 1)
        colored_objs.append(f'{article} {color} {noun}')

    if len(colored_objs) == 1:
        prompt = colored_objs[0]
    else:
        prompt = ', '.join(colored_objs[:-1]) + ' and ' + colored_objs[-1]

    return prompt

In [ ]:
def generate_csv_color_binding(output_path, id_counter, colors):
    with open(output_path, mode='a', newline='') as csv_file:
        writer = csv.writer(csv_file)

        # Write header only if starting fresh
        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt', 'obj1', 'bbox1', 'obj2', 'bbox2', 'obj3', 'bbox3', 'obj4', 'bbox4'])

        for num_objs in [1, 2, 3, 4]:
            print(f"Generating prompts with {num_objs} objects (with colors)...")
            min_size, max_size = NO_OVERLAPPING_RANGES[num_objs]

            if num_objs == 1:
                combinations = [[obj] for obj in obj_with_articles]
            else:
                combinations = list(itertools.combinations(obj_with_articles, num_objs))

            random.shuffle(combinations)

            total_needed = prompts_per_diff_level
            repeats_needed = math.ceil(total_needed / len(combinations))
            used_combinations = (combinations * repeats_needed)[:total_needed]

            for objs in tqdm(used_combinations):
                boxes = generate_non_overlapping_boxes(num_objs, min_size, max_size)

                # Randomly sample N colors, all different
                colors_sampled = random.sample(colors, num_objs)

                prompt = build_color_prompt(objs, colors_sampled)

                id_str = str(id_counter).zfill(4)
                category = 'color_binding'
                row = [id_str, category, prompt]

                for obj, box, color in itertools.zip_longest(objs, boxes, colors_sampled, fillvalue=''):
                    article, noun = obj.split(' ', 1)
                    obj_with_color = f'{article} {color} {noun}'

                    row.append(obj_with_color)

                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1

    return id_counter


### Attribute binding functions

Still no overlapping bounding boxes

In [ ]:
def build_attribute_prompt(objs, attribute_sampled):
    attribute_objs = []
    for obj, attribute in zip(objs, attribute_sampled):
        article, noun = obj.split(' ', 1)
        attribute_objs.append(f'{article} {attribute} {noun}')

    if len(attribute_objs) == 1:
        prompt = attribute_objs[0]
    else:
        prompt = ', '.join(attribute_objs[:-1]) + ' and ' + attribute_objs[-1]

    return prompt

In [ ]:
def generate_csv_attribute_binding(output_path, id_counter, attributes):
    with open(output_path, mode='a', newline='') as csv_file:
        writer = csv.writer(csv_file)

        # Write header only if starting fresh
        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt', 'obj1', 'bbox1', 'obj2', 'bbox2', 'obj3', 'bbox3', 'obj4', 'bbox4'])

        for num_objs in [1, 2, 3, 4]:
            print(f"Generating prompts with {num_objs} objects (with attributes)...")
            min_size, max_size = NO_OVERLAPPING_RANGES[num_objs]

            if num_objs == 1:
                combinations = [[obj] for obj in obj_with_articles]
            else:
                combinations = list(itertools.combinations(obj_with_articles, num_objs))

            random.shuffle(combinations)

            total_needed = prompts_per_diff_level
            repeats_needed = math.ceil(total_needed / len(combinations))
            used_combinations = (combinations * repeats_needed)[:total_needed]

            for objs in tqdm(used_combinations):
                boxes = generate_non_overlapping_boxes(num_objs, min_size, max_size)

                # Randomly sample N attributes, all different
                attributes_sampled = random.sample(attributes, num_objs)

                prompt = build_attribute_prompt(objs, attributes_sampled)

                id_str = str(id_counter).zfill(4)
                category = 'attribute_binding'
                row = [id_str, category, prompt]

                for obj, box, attribute in itertools.zip_longest(objs, boxes, attributes_sampled, fillvalue=''):
                    article, noun = obj.split(' ', 1)
                    obj_with_attribute = f'{article} {attribute} {noun}'

                    row.append(obj_with_attribute)

                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1

    return id_counter


### Overlapping bounding boxes functions

No more contraints on the dimension of the boxes since now they must overlap.



**Overlapping boxes generation ⬇️:**

- choose the first box randomly. The size will be between 10x10 and 510x510
- the other boxes are chosen randomly so that they overlap with at least one of the already generated boxes by picking an initial point inside of one of them

In [ ]:
def generate_overlapping_boxes(num_objs, min_size=80, max_size=510):
    boxes = []

    # First box chosen randomly from 10x10 to a max of 510x510
    w = random.randint(min_size, max_size)
    h = random.randint(min_size, max_size)

    x1 = random.randint(0, 512 - w)
    y1 = random.randint(0, 512 - h)

    box1 = (x1, y1, x1 + w, y1 + h)
    boxes.append(box1)

    for _ in range(1, num_objs):
        # Random size for new box
        w = random.randint(min_size, max_size)
        h = random.randint(min_size, max_size)

        # Pick one of the boxes already generated so far
        target_box = random.choice(boxes)

        # Pick a random point inside that target box
        overlap_x1 = random.randint(target_box[0], target_box[2] - 1)
        overlap_y1 = random.randint(target_box[1], target_box[3] - 1)

        # Now choose x1_new/y1_new so that box fully fits inside image
        # and contains the overlap point

        # For x1_new:
        x1_min = max(0, overlap_x1 - (w - 1))
        x1_max = min(overlap_x1, 512 - w)
        x1_new = random.randint(x1_min, x1_max)

        # For y1_new:
        y1_min = max(0, overlap_y1 - (h - 1))
        y1_max = min(overlap_y1, 512 - h)
        y1_new = random.randint(y1_min, y1_max)

        # Now the box is guaranteed to fit and to be >= 80x80
        x2_new = x1_new + w
        y2_new = y1_new + h

        box_new = (x1_new, y1_new, x2_new, y2_new)
        boxes.append(box_new)

    return boxes


In [ ]:
def generate_csv_overlapping_bboxes(output_path, id_counter):
    mode = 'w' if id_counter == 0 else 'a'

    with open(output_path, mode=mode, newline='') as csv_file:
        writer = csv.writer(csv_file)

        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt',
                             'obj1', 'bbox1',
                             'obj2', 'bbox2',
                             'obj3', 'bbox3',
                             'obj4', 'bbox4'])

        for num_objs in [1, 2, 3, 4]:
            print(f"Generating prompts with {num_objs} objects (with overlapping bboxes)...")
            min_size = 80
            max_size = 510

            if num_objs == 1:
                combinations = [[obj] for obj in obj_with_articles]
            else:
                combinations = list(itertools.combinations(obj_with_articles, num_objs))

            random.shuffle(combinations)

            total_needed = prompts_per_diff_level
            repeats_needed = math.ceil(total_needed / len(combinations))
            used_combinations = (combinations * repeats_needed)[:total_needed]

            for objs in tqdm(used_combinations):
                boxes = generate_overlapping_boxes(num_objs, min_size, max_size)

                id_str = str(id_counter).zfill(4)
                category = 'overlapping_bboxes'
                prompt = build_prompt(objs)

                row = [id_str, category, prompt]

                for obj, box in itertools.zip_longest(objs, boxes, fillvalue=''):
                    row.append(obj)
                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1

    return id_counter


### Small bounding boxes functions

In [ ]:
def generate_box_with_area_constraints(min_area, max_area, max_attempts=100):
    for _ in range(max_attempts):
        area = random.randint(min_area, max_area)
        aspect_ratio = random.uniform(0.5, 2)
        h = int(round((area / aspect_ratio) ** 0.5))
        w = int(round(h * aspect_ratio))
        if w <= IMAGE_SIZE and h <= IMAGE_SIZE:
            x = random.randint(0, IMAGE_SIZE - w)
            y = random.randint(0, IMAGE_SIZE - h)
            return (x, y, x + w, y + h)

def generate_non_overlapping_boxes_with_area(num_boxes, min_area, max_area, max_attempts=1000):
    boxes = []
    attempts = 0
    while len(boxes) < num_boxes and attempts < max_attempts:
        new_box = generate_box_with_area_constraints(min_area, max_area)
        if all(not boxes_overlap(new_box, existing) for existing in boxes):
            boxes.append(new_box)
        attempts += 1
    if len(boxes) < num_boxes:
        raise RuntimeError(f"Could not generate {num_boxes} non-overlapping boxes in time.")
    return boxes


In [ ]:
def generate_csv_small_bboxes(output_path, id_counter):
    mode = 'w' if id_counter == 0 else 'a'

    with open(output_path, mode=mode, newline='') as csv_file:
        writer = csv.writer(csv_file)

        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt',
                             'obj1', 'bbox1',
                             'obj2', 'bbox2',
                             'obj3', 'bbox3',
                             'obj4', 'bbox4'])

        for num_objs in [1, 2, 3, 4]:
            print(f"Generating prompts with {num_objs} objects (small bboxes)...")

            image_area = IMAGE_SIZE * IMAGE_SIZE

            min_area = int(image_area * 0.03)
            max_area = int(image_area * 0.10)

            if num_objs == 1:
                combinations = [[obj] for obj in obj_with_articles]
            else:
                combinations = list(itertools.combinations(obj_with_articles, num_objs))

            random.shuffle(combinations)

            total_needed = prompts_per_diff_level
            repeats_needed = math.ceil(total_needed / len(combinations))
            used_combinations = (combinations * repeats_needed)[:total_needed]

            for objs in tqdm(used_combinations):
                # IMPORTANT → generate small non-overlapping boxes
                boxes = generate_non_overlapping_boxes_with_area(num_objs, min_area, max_area)

                id_str = str(id_counter).zfill(4)
                category = 'small_bboxes'
                prompt = build_prompt(objs)

                row = [id_str, category, prompt]

                for obj, box in itertools.zip_longest(objs, boxes, fillvalue=''):
                    row.append(obj)
                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1

    return id_counter


### Object relations

In [ ]:
def generate_boxes_for_relation(obj1, obj2, relation, min_size, max_size):
    for attempt in range(400):
        corner = '' # needed if relation is far from
        on_offset = 0 # needed if relation is on
        # Generate first box conditioned on the relation
        if relation in ['to the left of']:
            # something will be on the right of the first box → leave space on the right
            w1 = random.randint(min_size, min(max_size, 512 - min_size))
            h1 = random.randint(min_size, max_size)
            x1 = random.randint(0, 512 - w1 - min_size)  # leave space on right (- min_size)
            y1 = random.randint(0, 512 - h1)

        elif relation in ['to the right of']:
            # something will be on the left of the first box → leave space on the left
            w1 = random.randint(min_size, min(max_size, 512 - min_size))
            h1 = random.randint(min_size, max_size)
            x1 = random.randint(min_size, 512 - w1)  # leave space on left (minimum random is min_size)
            y1 = random.randint(0, 512 - h1)

        elif relation in ['below', 'under']:
            # Place box1 in lower part of image, leaving space above for box2
            w1 = random.randint(min_size, max_size)
            h1 = random.randint(min_size, min(max_size, 512 - min_size))
            x1 = random.randint(0, 512 - w1)
            y1 = random.randint(min_size, 512 - h1) # Ensure space above (minimum random is min_size)

        elif relation in ['above', 'over']:
            # Place box1 in upper part of image, leaving space below for box2
            w1 = random.randint(min_size, max_size)
            h1 = random.randint(min_size, min(max_size, 512 - min_size))
            x1 = random.randint(0, 512 - w1)
            y1 = random.randint(0, 512 - h1 - min_size) # Ensure space below (- min_size)

        elif relation in ['on']:
            on_offset = random.randint(5, 30)
            # Place box1 in upper part of image, leaving space below for box2
            w1 = random.randint(min_size, max_size)
            h1 = random.randint(min_size, min(max_size, 512 - min_size - on_offset))
            x1 = random.randint(0, 512 - w1)
            y1 = random.randint(0, 512 - h1 - min_size) # Ensure space below (- min_size)

        elif relation in ['far from']:
            # select width and height for box1
            w1 = random.randint(min_size, max_size)
            h1 = random.randint(min_size, max_size)

            # define image halves
            mid_x, mid_y = 256, 256

            # choose a random corner
            corner = random.choice(['top left', 'top right', 'bottom left', 'bottom right'])

            if corner == 'top left':
                x1 = random.randint(0, mid_x - w1)
                y1 = random.randint(0, mid_y - h1)
            elif corner == 'top right':
                x1 = random.randint(mid_x, 512 - w1)
                y1 = random.randint(0, mid_y - h1)
            elif corner == 'bottom left':
                x1 = random.randint(0, mid_x - w1)
                y1 = random.randint(mid_y, 512 - h1)
            elif corner == 'bottom right':
                x1 = random.randint(mid_x, 512 - w1)
                y1 = random.randint(mid_y, 512 - h1)

        else:
            # near, next to, beside → fully random, let's leave everything to the probability
            w1 = random.randint(min_size, max_size)
            h1 = random.randint(min_size, max_size)
            x1 = random.randint(0, 512 - w1)
            y1 = random.randint(0, 512 - h1)

        box1 = (x1, y1, x1 + w1, y1 + h1)

        # Now generate box2 according to relation
        if relation in ['to the left of']:
            available_width = 512 - box1[2]
            if available_width < min_size:
                continue  # not enough space

            w2 = random.randint(min_size, available_width)
            h2 = random.randint(min_size, max_size)

            xmin = box1[2]
            xmax = 512 - w2
            if xmin > xmax:
                continue  # invalid range
            x2 = random.randint(xmin, xmax)
            y2 = random.randint(0, 512 - h2)

        elif relation in ['to the right of']:
            available_width = box1[0]
            if available_width < min_size:
                continue  # not enough space

            w2 = random.randint(min_size, available_width)
            h2 = random.randint(min_size, max_size)

            xmin = 0
            xmax = box1[0] - w2
            if xmin > xmax:
                continue  # invalid range
            x2 = random.randint(xmin, xmax)
            y2 = random.randint(0, 512 - h2)

        elif relation in ['below', 'under']:
            available_height = box1[1]
            if available_height < min_size:
                continue  # not enough space

            h2 = random.randint(min_size, available_height)
            w2 = random.randint(min_size, max_size)

            ymin = 0
            ymax = box1[1] - h2
            if ymin > ymax:
                continue  # invalid range
            y2 = random.randint(ymin, ymax)
            x2 = random.randint(0, 512 - w2)

        elif relation in ['above', 'over']:
            # box1 is in the top part of the image — box2 should be below
            available_height = 512 - box1[3]
            if available_height < min_size:
                continue  # not enough space

            h2 = random.randint(min_size, available_height)
            w2 = random.randint(min_size, max_size)

            ymin = box1[3]
            ymax = 512 - h2
            if ymin > ymax:
                continue  # invalid range
            y2 = random.randint(ymin, ymax)
            x2 = random.randint(0, 512 - w2)

        elif relation in [ 'on']:
            # box1 is in the top part of the image — box2 should be below
            available_space = 512 - box1[3] - on_offset

            if(available_space < min_size):
                continue  # not enough space

            w2 = random.randint(min_size, max_size)
            h2 = random.randint(min_size, available_space)
            x2 = random.randint(max(0, box1[0] - w2), min(512 - w2, box1[0] + w2)) # limit the chosing of x so that box2 is somehow under box1
            y2 = box1[3] + on_offset


        elif relation in ['far from']:
            # position box2 in the furthest corner from box1
            w2 = random.randint(min_size, max_size)
            h2 = random.randint(min_size, max_size)

            if corner == 'top left':
                x2 = random.randint(256, 512 - w2)
                y2 = random.randint(256, 512 - h2)
            elif corner == 'top right':
                x2 = random.randint(0, 256 - w2)
                y2 = random.randint(256, 512 - h2)
            elif corner == 'bottom left':
                x2 = random.randint(256, 512 - w2)
                y2 = random.randint(0, 256 - h2)
            elif corner == 'bottom right':
                x2 = random.randint(0, 256 - w2)
                y2 = random.randint(0, 256 - h2)

        elif relation in ['near']:
            # Place box2 near box1, all directions that have enough space are okay, no overlapping
            # step1: calculate all directions from box1 that have enough space (i.e. that have at least min_size + offset from the border)
            # step2: chose a random direction from the ones that can be used
            # step3: place box2 in that space
            offset = random.randint(5, 30)
            directions = []

            # Determine which directions have enough space for box2
            if box1[0] >= min_size + offset:
                directions.append('left')
            if box1[2] + min_size + offset <= 512:
                directions.append('right')
            if box1[1] >= min_size + offset:
                directions.append('up')
            if box1[3] + min_size + offset <= 512:
                directions.append('down')

            if not directions:
                # If somehow no direction is available, try again
                continue
            else:
                direction = random.choice(directions)

                if direction == 'left': # on the left of box1
                    available_space = box1[0] - offset
                    w2 = random.randint(min_size, available_space)
                    h2 = random.randint(min_size, max_size)
                    x2 = box1[0] - w2 - offset
                    y2 = random.randint(max(0, box1[1] - h2), min(512 - h2, box1[1] + h2)) # limit the chosing of y so that they are actually next to each other
                elif direction == 'right': # on the rigth of box1
                    available_space = 512 - box1[2] - offset
                    w2 = random.randint(min_size, available_space)
                    h2 = random.randint(min_size, max_size)
                    x2 = box1[2] + offset
                    y2 = random.randint(max(0, box1[1] - h2), min(512 - h2, box1[1] + h2))
                elif direction == 'up': # above box1
                    available_space = box1[1] - offset
                    w2 = random.randint(min_size, max_size)
                    h2 = random.randint(min_size, available_space)
                    x2 = random.randint(max(0, box1[0] - w2), min(512 - w2, box1[0] + w2))
                    y2 = box1[1] - h2 - offset
                elif direction == 'down': # below box1
                    available_space = 512 - box1[3] - offset
                    w2 = random.randint(min_size, max_size)
                    h2 = random.randint(min_size, available_space)
                    x2 = random.randint(max(0, box1[0] - w2), min(512 - w2, box1[0] + w2)) # limit the chosing of x so that box2 is somehow under box1
                    y2 = box1[3] + offset

        elif relation in ['next to', 'beside']:
            # Place box2 near box1 but only left or right of each other, no overlapping
            offset = random.randint(5, 30)
            directions = []

            # Determine which directions have enough space for box2
            if box1[0] >= min_size + offset:
                directions.append('left')
            if box1[2] + min_size + offset <= 512:
                directions.append('right')

            if not directions:
                # If somehow no direction is available, try again
                continue
            else:
                direction = random.choice(directions)

                if direction == 'left': # on the left of box1
                    available_space = box1[0] - offset
                    w2 = random.randint(min_size, available_space)
                    h2 = random.randint(min_size, max_size)
                    x2 = box1[0] - w2 - offset
                    y2 = random.randint(max(0, box1[1] - h2), min(512 - h2, box1[1] + h2)) # limit the chosing of y so that they are actually next to each other
                elif direction == 'right': # on the rigth of box1
                    available_space = 512 - box1[2] - offset
                    w2 = random.randint(min_size, available_space)
                    h2 = random.randint(min_size, max_size)
                    x2 = box1[2] + offset
                    y2 = random.randint(max(0, box1[1] - h2), min(512 - h2, box1[1] + h2))

        else:
            # Fully random for other relations
            w2 = random.randint(min_size, max_size)
            h2 = random.randint(min_size, max_size)
            x2 = random.randint(0, 512 - w2)
            y2 = random.randint(0, 512 - h2)

        box2 = (x2, y2, x2 + w2, y2 + h2)


        if not boxes_overlap(box1, box2):
            return [box1, box2]

    else:
        # If after 1000 attempts we couldn't place box2, retry placing box1 and box2
        print(f"Warning: could not find valid box2 for relation '{relation}' after 400 attempts.")
        return None

    return [box1, box2]

In [ ]:
def build_relationship_prompt(objs, relations):
    if len(objs) == 2:
        return f"{objs[0]} {relations[0]} {objs[1]}"
    elif len(objs) == 4:
        return f"{objs[0]} {relations[0]} {objs[1]} and {objs[2]} {relations[1]} {objs[3]}"
    else:
        raise ValueError("Only 2 or 4 objects supported")


In [ ]:
def generate_csv_object_relationship(output_path, id_counter, spatial_relations):
    mode = 'w' if id_counter == 0 else 'a'

    with open(output_path, mode=mode, newline='') as csv_file:
        writer = csv.writer(csv_file)

        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt',
                             'obj1', 'bbox1',
                             'obj2', 'bbox2',
                             'obj3', 'bbox3',
                             'obj4', 'bbox4'])

        for num_objs in [2, 4]:
            print(f"Generating prompts with {num_objs} objects (object_relationship)...")
            total_needed = prompts_per_diff_level

            min_size, max_size = NO_OVERLAPPING_RANGES[num_objs]

            if num_objs == 2:
                combinations = [random.sample(obj_with_articles, 2) for _ in range(total_needed)]
            else:
                combinations = [random.sample(obj_with_articles, 4) for _ in range(total_needed)]

            random.shuffle(combinations)

            repeats_needed = math.ceil(total_needed / len(combinations))
            used_combinations = (combinations * repeats_needed)[:total_needed]

            failure_count = 0
            successful = 0
            attempts = 0
            max_total_attempts = prompts_per_diff_level * 20  # avoid infinite loop

            while successful < prompts_per_diff_level and attempts < max_total_attempts:
                attempts += 1
                objs = random.sample(obj_with_articles, num_objs)
                relations = random.choices(spatial_relations, k=num_objs // 2)
                boxes = []

                if num_objs == 2:
                    result = generate_boxes_for_relation(objs[0], objs[1], relations[0], min_size, max_size)
                    if result is None:
                        continue
                    boxes += result

                elif num_objs == 4:
                    result1 = generate_boxes_for_relation(objs[0], objs[1], relations[0], min_size, max_size)
                    if result1 is None:
                        continue

                    for _ in range(10):  # max_attempts
                        result2 = generate_boxes_for_relation(objs[2], objs[3], relations[1], min_size, max_size)
                        if result2 is None:
                            continue
                        b3, b4 = result2
                        if not any(boxes_overlap(b, b3) or boxes_overlap(b, b4) for b in result1):
                            boxes = result1 + result2
                            break
                    else:
                        continue  # all 10 attempts failed, retry outer loop

                # build prompt and write
                id_str = str(id_counter).zfill(4)
                category = 'object_relationship'
                prompt = build_relationship_prompt(objs, relations)

                row = [id_str, category, prompt]
                for obj, box in itertools.zip_longest(objs, boxes, fillvalue=''):
                    row.append(obj)
                    if box:
                        box_str = f'{box[0]},{box[1]},{box[2]},{box[3]}'
                    else:
                        box_str = ''
                    row.append(box_str)

                while len(row) < 11:
                    row.append('')

                writer.writerow(row)
                id_counter += 1
                successful += 1

    return id_counter



### Complex composition functions

First, let's load the csv file containing the complex prompts.
To follow the rest of the tasks, the csv should contain a total of 2048 prompts, 512 for each level of difficulty (i.e. for each number of objects 1, 2, 3 or 4).

The csv should have the following structure:


```
prompt, objects
```
here an example:

```
prompt,objects
A happy horse is behind a white toolbox.,"a happy horse, a white toolbox"
A small guitar is in front of a red pan.,"a small guitar, a red pan"
A green bottle is between a white skateboard.,"a green bottle, a white skateboard"
```

I generated them using chatGPT with the following prompts:



---


For 1 obj prompts:


```
## 1 object prompt:

Generate 512 natural compositional phrases with various structures and creativity.
Each prompt must describe one unique object from the list below. Use all the objects at least one time.
Each object should be enriched with at least one descriptive attribute that refers to its color, shape, material, appearance, or dimension. Attributes can be qualitative (e.g., "fluffy", "shiny") or refer to physical characteristics (e.g., "large", "tall").
Each prompt should consist of a short, vivid sentence that situates the object in a dynamic or descriptive context, similar to the following examples:
A fish swam swiftly through the clear water.
A bear lumbered through the dense forest.
A tulip bloomed brightly in the garden.
Avoid passive or overly generic constructions—aim for imaginative, specific scenarios.
Adjust the articles of the objects if needed.

Object List (use each exactly once):
obj_with_articles = [
'a rose', 'an oak', 'a beetle', 'a skyscraper', 'a tree', 'a baby', 'a bed', 'a lamp', 'a dog', 'a laptop',
'a bicycle', 'a person', 'a car', 'a bus', 'a cat', 'a book', 'a chair', 'a boy', 'a couch', 'a table',
'a plant', 'a toilet', 'a cellphone', 'a microwave', 'a sheep', 'a boat', 'a banana', 'a stop sign',
'a donut', 'a cow', 'a clock', 'a bottle', 'an umbrella', 'a bird', 'a guitar', 'a toothbrush', 'a parking meter',
'a bench', 'a platypus', 'a keyboard', 'a baseball bat', 'a vase', 'a surfboard', 'a tiger', 'a train', 'a flower',
'a sandwich', 'a spoon', 'a pizza', 'a carrot', 'a teddy bear', 'an hot-dog', 'a skateboard', 'a kite', 'a broom',
'an apple', 'a handbag', 'a horse', 'a snowboard', 'a giraffe', 'a tie', 'a shower', 'a traffic light', 'a bear',
'a toaster', 'a knife', 'a baseball glove', 'a crocodile', 'a suitcase', 'a fork', 'a cake', 'a cup', 'a bowl',
'a hair drier', 'an elephant', 'a mouse', 'a mushroom', 'a motorcycle', 'a turtle', 'a tennis racket', 'a truck',
'a zebra', 'a fire hydrant', 'an oven', 'a sink', 'a frisbee', 'a hat', 'a ruler', 'a shoe', 'a ball', 'a candle',
'a ladder', 'a charger', 'a mug', 'a tape', 'a shirt', 'a pillow', 'a pan', 'a plate', 'a shampoo', 'a hammer',
'a blender', 'a basket', 'a screwdriver', 'a wallet', 'a bin', 'a leaf', 'a bucket', 'a monitor', 'a watch',
'a flashlight', 'a sock', 'a door', 'a scarf', 'a speaker', 'a desk', 'a backpack', 'a printer', 'a remote',
'a glass', 'a curtain', 'a toolbox', 'a drill', 'a notebook', 'a television', 'a soap', 'a ring', 'a refrigerator'
]

Allowed Attributes:
attributes = [
'aggressive', 'black', 'blue', 'bright', 'clean', 'crowded', 'dark', 'fast', 'fluffy', 'fuzzy', 'green', 'happy',
'large', 'pink', 'red', 'rotten', 'rough', 'shiny', 'short', 'silver', 'small', 'smooth', 'snowy', 'soft',
'tall', 'warm', 'white', 'wooden', 'yellow'
]

Allowed Colors:
colors = ['black', 'blue', 'brown', 'gray', 'green', 'pink', 'purple', 'red', 'white', 'yellow', 'orange']

Output Format:
Return the result as a CSV with two columns:
prompt, object1
Each row should contain:
The generated sentence
The noun chunk used

Example output format:
prompt,object1
A blue fish swam swiftly through the clear water, a blue fish
A large brown bear lumbered through the dense forest, a large brown bear
A pink tulip bloomed brightly in the garden, a pink tulip

The prompts should be inside quotes if needed to keep the correct numbers of columns in the csv.
In the sentence, keep noun chunks unbroken—adjectives modifying a noun should not be split by commas. Treat the entire noun chunk as a single unit (e.g., "a small red ball", not "a small, red ball").
The articles should remain consistent between the prompt and the noun chunk.



```
---
For 2,3,4 objects prompts:


```
# 2,3,4 objects prompt:

Generate 512 natural compositional phrases with various structures and creativity.
Each prompt must describe a scene involving exactly 4 unique objects.
Objects can be reused across multiple prompts, but each object must appear only once within any given prompt.
Each object must be enriched with at least one descriptive attribute, which may describe:
Color, shape, material, appearance, or dimension
Or a spatial relation between objects in the same prompt
Each prompt should be a short, vivid sentence that situates the objects in a dynamic or descriptive context, similar to the following examples:
A bright pink flower swayed gently under the tall oak tree.
A black laptop rested beside a green coffee mug on the messy desk.

Avoid passive or overly generic constructions—aim for imaginative, specific scenarios.
Adjust the articles of the objects if needed.

Object List
(You can freely reuse any of these objects across different prompts, but not within the same prompt.)
obj_with_articles = [
'a rose', 'an oak', 'a beetle', 'a skyscraper', 'a tree', 'a baby', 'a bed', 'a lamp', 'a dog', 'a laptop',
'a bicycle', 'a person', 'a car', 'a bus', 'a cat', 'a book', 'a chair', 'a boy', 'a couch', 'a table',
'a plant', 'a toilet', 'a cellphone', 'a microwave', 'a sheep', 'a boat', 'a banana', 'a stop sign',
'a donut', 'a cow', 'a clock', 'a bottle', 'an umbrella', 'a bird', 'a guitar', 'a toothbrush', 'a parking meter',
'a bench', 'a platypus', 'a keyboard', 'a baseball bat', 'a vase', 'a surfboard', 'a tiger', 'a train', 'a flower',
'a sandwich', 'a spoon', 'a pizza', 'a carrot', 'a teddy bear', 'an hot-dog', 'a skateboard', 'a kite', 'a broom',
'an apple', 'a handbag', 'a horse', 'a snowboard', 'a giraffe', 'a tie', 'a shower', 'a traffic light', 'a bear',
'a toaster', 'a knife', 'a baseball glove', 'a crocodile', 'a suitcase', 'a fork', 'a cake', 'a cup', 'a bowl',
'a hair drier', 'an elephant', 'a mouse', 'a mushroom', 'a motorcycle', 'a turtle', 'a tennis racket', 'a truck',
'a zebra', 'a fire hydrant', 'an oven', 'a sink', 'a frisbee', 'a hat', 'a ruler', 'a shoe', 'a ball', 'a candle',
'a ladder', 'a charger', 'a mug', 'a tape', 'a shirt', 'a pillow', 'a pan', 'a plate', 'a shampoo', 'a hammer',
'a blender', 'a basket', 'a screwdriver', 'a wallet', 'a bin', 'a leaf', 'a bucket', 'a monitor', 'a watch',
'a flashlight', 'a sock', 'a door', 'a scarf', 'a speaker', 'a desk', 'a backpack', 'a printer', 'a remote',
'a glass', 'a curtain', 'a toolbox', 'a drill', 'a notebook', 'a television', 'a soap', 'a ring', 'a refrigerator'
]

Allowed Attributes (for appearance or material):
attributes = [
'aggressive', 'black', 'blue', 'bright', 'clean', 'crowded', 'dark', 'fast', 'fluffy', 'fuzzy', 'green', 'happy',
'large', 'pink', 'red', 'rotten', 'rough', 'shiny', 'short', 'silver', 'small', 'smooth', 'snowy', 'soft',
'tall', 'warm', 'white', 'wooden', 'yellow'
]

Allowed Colors (subset of attributes):
colors = ['black', 'blue', 'brown', 'gray', 'green', 'pink', 'purple', 'red', 'white', 'yellow', 'orange']

Spatial Relations (to be used as part of attributes or composition logic):
spatial_relations = [
'on top of', 'beside', 'under', 'above', 'next to', 'beneath', 'behind', 'in front of', 'between', 'leaning on',
'inside', 'resting on', 'attached to', 'surrounded by', 'placed near'
]

Output Format:
Return the result as a CSV with two columns:
prompt, object1, object2, ..
Each row should contain:
The generated sentence
The noun chunks used

Example output format (for N=2):
prompt,object1,object2
A fluffy cat jumped onto the soft couch near the window" ,a fluffy cat, a soft couch
A red bicycle leaned against a wooden bench in the park ,a red bicycle, a wooden bench

The prompts should be inside quotes if needed to keep the correct numbers of columns in the csv.
In the sentence, keep noun chunks unbroken—adjectives modifying a noun should not be split by commas. Treat the entire noun chunk as a single unit (e.g., "a small red ball", not "a small, red ball").
The articles should remain consistent between the prompt and the noun chunks.

```







In [ ]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

In [ ]:
def extract_spatial_relations(prompt: str, local_prompts: list[str]):
    doc = nlp(prompt)
    relations = []

    # Convert to lowercase for fuzzy match
    local_prompts_lower = [lp.lower() for lp in local_prompts]

    for token in doc:
        # Skip if token is not a preposition or not in spatial list
        if token.dep_ == "prep" and token.text.lower() in spatial_relations:
            prep = token.text.lower()

            # Try to get the object of the preposition (e.g., "mouse" in "under the mouse")
            pobj = next((child for child in token.children if child.dep_ == "pobj"), None)
            # Head is usually the verb, go one level up to get noun subject
            subject = None
            if token.head:
                for child in token.head.children:
                    if child.dep_ in ("nsubj", "nsubjpass"):
                        subject = child

            if subject and pobj:
                # Try to match to full local_prompt strings
                subj_match = next((lp for lp in local_prompts_lower if subject.text.lower() in lp), None)
                obj_match = next((lp for lp in local_prompts_lower if pobj.text.lower() in lp), None)

                if subj_match and obj_match:
                    relations.append([subj_match, obj_match, prep])

    return relations

In [ ]:
def extract_object_list(row):
    return [
        str(row.get(f'object{i}', '')).strip()
        for i in range(1, 5)
        if pd.notna(row.get(f'object{i}', '')) and str(row.get(f'object{i}', '')).strip() != ''
    ]

complex_prompts_df["triplet"] = complex_prompts_df.apply(
    lambda row: extract_spatial_relations(row["prompt"], extract_object_list(row)),
    axis=1
)

In [ ]:
complex_prompts_df["triplet"] = complex_prompts_df.apply(lambda row: extract_spatial_relations(row["prompt"], extract_object_list(row)), axis=1)
complex_prompts_df.tail()

In [ ]:
import csv
import random
import math
from tqdm import tqdm
import pandas as pd

MIN_SIZE = int(0.03 * IMAGE_SIZE)
MAX_SIZE = int(0.97 * IMAGE_SIZE)

MIN_SIZE = 80
MAX_SIZE = 510

def random_bbox(min_size=MIN_SIZE, max_size=MAX_SIZE):
    """Generate a random bounding box (x1, y1, x2, y2) within image bounds."""
    width = random.randint(min_size, max_size)
    height = random.randint(min_size, max_size)
    x1 = random.randint(0, IMAGE_SIZE - width)
    y1 = random.randint(0, IMAGE_SIZE - height)
    x2 = x1 + width
    y2 = y1 + height
    assert x2 > x1 and y2 > y1, "Invalid bbox: negative width/height"
    return [x1, y1, x2, y2]

def detect_spatial_relations(prompt):
    """Return all spatial relations found in the prompt."""
    lowered = prompt.lower()
    return [rel for rel in spatial_relations if rel in lowered]

def generate_csv_complex_composition(df, output_path, id_counter):
    mode = 'w' if id_counter == 0 else 'a'

    with open(output_path, mode=mode, newline='') as csv_file:
        writer = csv.writer(csv_file)

        if id_counter == 0:
            writer.writerow(['id', 'category', 'prompt', 'obj1', 'bbox1', 'obj2', 'bbox2', 'obj3', 'bbox3', 'obj4', 'bbox4'])

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = row['prompt']
            relations = row['triplet']  # [['obj1', 'obj2', 'relation'], ...]

            # Get the object labels from the dataframe
            objects = [row.get(f'object{i}') for i in range(1, 5)]
            objects = [obj for obj in objects if pd.notna(obj) and str(obj).strip() != '']
            num_objs = len(objects)

            if num_objs == 0:
                continue

            # Assign boxes
            boxes = {}

            used_objects = set()
            if relations:
                for triplet in relations:
                    subj, obj, relation = triplet

                    if subj not in boxes or obj not in boxes:
                        try:
                            box1, box2 = generate_boxes_for_relation(subj, obj, relation, MIN_SIZE, MAX_SIZE)
                            boxes[subj] = box1
                            boxes[obj] = box2
                            used_objects.update([subj, obj])
                        except Exception as e:
                            # Fallback to random if there's any issue
                            print(f"Error generating boxes for {subj} and {obj}: {e}. Falling back to random.")
                            boxes[subj] = random_bbox()
                            boxes[obj] = random_bbox()
                            used_objects.update([subj, obj])

            # Assign random boxes to the remaining objects
            for obj in objects:
                if obj not in boxes:
                    boxes[obj] = random_bbox()

            # Build row
            id_str = str(id_counter).zfill(4)
            category = 'complex_composition'
            row_out = [id_str, category, prompt]

            for obj in objects:
                row_out.append(obj)
                box = boxes[obj]
                box_str = f"{box[0]},{box[1]},{box[2]},{box[3]}"
                row_out.append(box_str)

            while len(row_out) < 11:
                row_out.append('')

            writer.writerow(row_out)
            id_counter += 1

    return id_counter



### Generation of the full CSV and download

In [ ]:
id_counter = 0
output_path = 'extendedDataset.csv'

#### Object binding

In [ ]:
# object binding
id_counter = generate_csv_object_binding(output_path, id_counter)

#### Color binding

In [ ]:
# color binding
id_counter = generate_csv_color_binding(output_path, id_counter, colors)

#### Attribute binding

In [ ]:
# attribute binding
id_counter = generate_csv_attribute_binding(output_path, id_counter, attributes)

#### Overlapping bounding boxes

In [ ]:
# overlapping bounding boxes
id_counter = generate_csv_overlapping_bboxes(output_path, id_counter)

#### Small bounding boxes

In [ ]:
# small bounding boxes
id_counter = generate_csv_small_bboxes(output_path, id_counter)

#### Object relations

In [ ]:
# object relations
id_counter = generate_csv_object_relationship(output_path, id_counter, spatial_relations)

#### Complex composition

In [ ]:
# complex composition
id_counter = generate_csv_complex_composition(complex_prompts_df, output_path, id_counter)

#### Export file

Export the created prompt and boxes in a csv file

In [ ]:
from google.colab import files
files.download('extendedDataset.csv')

# Visualize boxes

In [ ]:
def parse_bbox(bbox_str):
    if pd.isna(bbox_str) or bbox_str == '':
        return None
    return tuple(map(int, bbox_str.split(',')))

Extract randomly the selected number of prompts with their bounding boxes and display them. Choose the preferred number of boxes per row.

## Visualize random prompts

In [ ]:
def visualize_csv_boxes_grid(csv_path, num_samples=35, boxes_per_row=5, random_seed=42):
    df = pd.read_csv(csv_path)
    sampled_rows = df.sample(n=num_samples, random_state=random_seed)

    num_cols = boxes_per_row
    num_rows = math.ceil(num_samples / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(boxes_per_row * 4, num_rows * 4))
    axes = axes.flatten()  # easy indexing even if last row is incomplete

    for ax, (idx, row) in zip(axes, sampled_rows.iterrows()):
        # Read the boxes
        boxes = []
        labels = []
        for i in range(1, 5):
            obj_col = f'obj{i}'
            bbox_col = f'bbox{i}'
            obj_name = row[obj_col]
            bbox = parse_bbox(row[bbox_col])
            if obj_name and isinstance(obj_name, str) and bbox:
                boxes.append(bbox)
                labels.append(obj_name)

        # Plot this row
        ax.set_xlim(0, 512)
        ax.set_ylim(0, 512)
        ax.invert_yaxis()

        for i, box in enumerate(boxes):
            x1, y1, x3, y3 = box
            width = x3 - x1
            height = y3 - y1
            rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)

            # Label
            ax.text(x1 + 3, y1 - 5, labels[i], color='blue', fontsize=8)

        ax.set_title(f'ID {row["id"]}\n{row["prompt"]}\n{row["category"]}', fontsize=10)
        ax.grid(True)

    # Hide unused axes (if num_samples is not multiple of boxes_per_row)
    for ax in axes[num_samples:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_csv_boxes_grid('extendedDataset.csv', num_samples=35, boxes_per_row=5, random_seed=42)

## Visualize prompts by id

In [ ]:
def visualize_csv_boxes_by_ids(csv_path, selected_ids, boxes_per_row=5):
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import math

    def parse_bbox(bbox_str):
        try:
            return list(map(int, bbox_str.split(',')))
        except:
            return None

    df = pd.read_csv(csv_path)

    # Make sure we match padded string IDs (e.g. '00005')
    padded_ids = [str(id_) for id_ in selected_ids]
    selected_rows = df[df['id'].astype(str).isin(padded_ids)]

    if selected_rows.empty:
        print("⚠️ No matching IDs found.")
        return

    num_samples = len(selected_rows)
    num_cols = boxes_per_row
    num_rows = math.ceil(num_samples / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(boxes_per_row * 4, num_rows * 4))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, selected_rows.iterrows()):
        boxes = []
        labels = []
        for i in range(1, 5):
            obj_col = f'obj{i}'
            bbox_col = f'bbox{i}'
            obj_name = row[obj_col]
            bbox = parse_bbox(row[bbox_col])
            if obj_name and isinstance(obj_name, str) and bbox:
                boxes.append(bbox)
                labels.append(obj_name)

        ax.set_xlim(0, 512)
        ax.set_ylim(0, 512)
        ax.invert_yaxis()

        for i, box in enumerate(boxes):
            x1, y1, x3, y3 = box
            width = x3 - x1
            height = y3 - y1
            rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1 + 3, y1 - 5, labels[i], color='blue', fontsize=8)

        ax.set_title(f'ID {row["id"]}\n{row["prompt"]}\n{row["category"]}', fontsize=10)
        ax.grid(True)

    for ax in axes[num_samples:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize specific prompts (e.g., ID 0, 5, 18, and 29)
visualize_csv_boxes_by_ids("extendedDataset.csv", selected_ids=[929])
